# Supplemental Material, Secs. I–II: analytical derivations

Manifold-Interleaved Spirals and Semiclassical Continuum Scattering — L. M. Chinellato, L. Woodland, R. Coldea, C. D. Batista

This notebook reproduces every analytical result of Sec. I (Eqs. S1–S7) and Secs. II A–F (Eqs. S8–S54) with SymPy.

**Conventions.** $J=S=1$ unless stated otherwise, $\bar\Delta\equiv1-\Delta$, and angles $\vartheta^\alpha$ are measured from $-\hat z$. In-plane components are ordered $(\parallel,\perp)$ with $A_\parallel=-\mathbf A\cdot\hat z$, so $\mathbf S_\alpha=(\cos\vartheta^\alpha,\sin\vartheta^\alpha)$. In code, $t\equiv\vartheta^{\rm A}$, $c=\cos t$, $s=\sin t$.

**Two strategies.**
1. *Exact identities* (Sec. I, parts of II A and II E) are proven by polynomial reduction modulo the relations that define $\Lambda$ and $\Gamma$. A zero remainder is a proof, not a numerical coincidence.
2. *Perturbative results* (Table II, Eqs. S28–S37, S44–S54) use truncated power series in $\bar\Delta$. Their coefficients are Laurent polynomials in $z=e^{it}$, so averages over a turn of the manifold are exact $z^0$ coefficients.



In [2]:
import sympy as sp
import numpy as np
from IPython.display import display, Math, Markdown

I = sp.I
w = sp.Rational(-1, 2) + I*sp.sqrt(3)/2          # omega = exp(2 pi i / 3)

def show(tag, lhs, rhs=None):
    body = sp.latex(lhs) if rhs is None else sp.latex(lhs) + " = " + sp.latex(rhs)
    display(Math(r"\textrm{" + tag + r"}\qquad " + body))

def check(label, expr):
    # Assert that expr vanishes identically.
    e = sp.expand(expr)
    if e != 0:
        e = sp.simplify(e)
    assert e == 0, (label, e)
    print("\u2713", label)

def trig_zero(e):
    # Exact zero test for trigonometric expressions via exponentials.
    return sp.expand(sp.powsimp(sp.expand(e.rewrite(sp.exp))))

---
## I. Ground-state manifold of a single layer

### I.A Parametrization, Eqs. (S1)–(S2)

In a three-sublattice state every site has one bond of each type AB, BC, CA, so the classical energy per site is

$$e=\sum_{\langle\alpha\beta\rangle}\big[\Delta\sin\vartheta^\alpha\sin\vartheta^\beta+\cos\vartheta^\alpha\cos\vartheta^\beta\big].$$

Eq. (S1) gives $\vartheta^{\rm B,C}=\pi+\epsilon\mp\gamma$ with

$$\cos\gamma=\frac{\Delta}{(1+\Delta)\Lambda},\quad \sin\gamma=\frac{\Gamma}{(1+\Delta)\Lambda},\qquad \cos\epsilon=\frac{\Delta c}{\Lambda},\quad \sin\epsilon=\frac{s}{\Lambda},$$

where $\Lambda=\sqrt{s^2+\Delta^2c^2}$ and $\Gamma=\sqrt{(1+\Delta)^2\Lambda^2-\Delta^2}$ [Eq. (S2)].

Two sign choices are fixed here:
- $\sin\gamma\ge0$, because $\gamma\in(0,\pi)$.
- $\sin\epsilon=s/\Lambda$, the branch continuous with $\epsilon\to t$ at $\Delta=1$.

With these, all quantities are rational in $(c,s,\Lambda,\Gamma)$. The relations $s^2+c^2=1$, $\Lambda^2=s^2+\Delta^2c^2$, $\Gamma^2=(1+\Delta)^2\Lambda^2-\Delta^2$ have pairwise coprime leading monomials in the lex order $\Gamma>\Lambda>s>c$. They therefore form a Gröbner basis, and the remainder of any polynomial modulo them is canonical.

In [3]:
D = sp.Symbol('Delta', positive=True)
c, s, L, G = sp.symbols('c s Lambda Gamma', real=True)
rels = [G**2 - ((1 + D)**2*L**2 - D**2), L**2 - (s**2 + D**2*c**2), s**2 + c**2 - 1]

def red(expr):
    # Canonical form of a rational function of (c, s, Lambda, Gamma) modulo the relations.
    num, den = sp.fraction(sp.together(sp.expand(expr)))
    rn = sp.reduced(sp.expand(num), rels, G, L, s, c)[1]
    rd = sp.reduced(sp.expand(den), rels, G, L, s, c)[1]
    return sp.factor(sp.cancel(rn/rd))

# d/dt on the manifold: c' = -s, s' = c, and the implicit derivatives of Lambda and Gamma
Lp = (1 - D**2)*s*c/L                 # from Lambda^2 = s^2 + Delta^2 c^2
Gp = (1 + D)**2*L*Lp/G                # from Gamma^2 = (1+Delta)^2 Lambda^2 - Delta^2
def ddt(f):
    return sp.diff(f, c)*(-s) + sp.diff(f, s)*c + sp.diff(f, L)*Lp + sp.diff(f, G)*Gp

cos_g, sin_g = D/((1 + D)*L), G/((1 + D)*L)
cos_e, sin_e = D*c/L, s/L
# theta^{B,C} = pi + eps -/+ gamma
cosT = {'A': c, 'B': -(cos_e*cos_g + sin_e*sin_g), 'C': -(cos_e*cos_g - sin_e*sin_g)}
sinT = {'A': s, 'B': -(sin_e*cos_g - cos_e*sin_g), 'C': -(sin_e*cos_g + cos_e*sin_g)}
pairs = [('A', 'B'), ('B', 'C'), ('C', 'A')]

**Checks.** The spins have unit length. The energy is the same for every member. The manifold consists of critical points: $\partial e/\partial\vartheta^\alpha=0$ for all three angles independently.

In [4]:
for a in 'ABC':
    check(f"|S_{a}| = 1", red(cosT[a]**2 + sinT[a]**2 - 1))

e = sum(D*sinT[a]*sinT[b] + cosT[a]*cosT[b] for a, b in pairs)
e_manifold = red(e)
show("energy per site on the manifold:", sp.Symbol('e'), e_manifold)
check("e = -(1+Delta+Delta^2)/(1+Delta)", e_manifold + (1 + D + D**2)/(1 + D))

for a in 'ABC':
    grad = sum(D*cosT[a]*sinT[b] - sinT[a]*cosT[b] for b in 'ABC' if b != a)
    check(f"de/dtheta^{a} = 0", red(grad))

✓ |S_A| = 1
✓ |S_B| = 1
✓ |S_C| = 1


<IPython.core.display.Math object>

✓ e = -(1+Delta+Delta^2)/(1+Delta)
✓ de/dtheta^A = 0
✓ de/dtheta^B = 0
✓ de/dtheta^C = 0


Stationarity and equal energy show that Eq. (S1) is a family of degenerate critical points. That it is the global minimum is the result of Ref. [S1]; a numerical check follows.

In [5]:
from scipy.optimize import minimize

def e_num(th, Dv):
    return sum(Dv*np.sin(th[i])*np.sin(th[j]) + np.cos(th[i])*np.cos(th[j]) for i, j in [(0, 1), (1, 2), (2, 0)])

rng = np.random.default_rng(0)
for Dv in (0.3, 0.635, 0.9):
    best = min(minimize(e_num, rng.uniform(0, 2*np.pi, 3), args=(Dv,)).fun for _ in range(200))
    print(f"Delta={Dv}:  global min (200 starts) = {best:.12f},   -(1+D+D^2)/(1+D) = {-(1+Dv+Dv**2)/(1+Dv):.12f}")

Delta=0.3:  global min (200 starts) = -1.069230769231,   -(1+D+D^2)/(1+D) = -1.069230769231
Delta=0.635:  global min (200 starts) = -1.246620795107,   -(1+D+D^2)/(1+D) = -1.246620795107
Delta=0.9:  global min (200 starts) = -1.426315789474,   -(1+D+D^2)/(1+D) = -1.426315789474


**Isotropic limit.** At $\Delta=1$: $\Lambda=1$, $\Gamma=\sqrt3$, so $\epsilon=t$, $\gamma=\pi/3$, and $\vartheta^{\rm B,C}=t\pm2\pi/3$. The manifold becomes the orbit of the rigid $120^\circ$ state.

In [6]:
t = sp.Symbol('t', real=True)
iso = {D: 1, L: 1, G: sp.sqrt(3)}
check("cos gamma = 1/2 at Delta=1", cos_g.subs(iso) - sp.Rational(1, 2))
for a, shift in (('B', 2*sp.pi/3), ('C', 4*sp.pi/3)):
    lhs = (cosT[a] + I*sinT[a]).subs(iso).subs({c: sp.cos(t), s: sp.sin(t)})
    check(f"exp(i theta^{a}) = exp(i(t + {shift}))", sp.expand_complex(lhs - sp.exp(I*(t + shift))))

✓ cos gamma = 1/2 at Delta=1
✓ exp(i theta^B) = exp(i(t + 2*pi/3))
✓ exp(i theta^C) = exp(i(t + 4*pi/3))


### I.B Magnetization angle, Eq. (S3)

Let $Z\equiv M_\parallel+iM_\perp=\sum_\alpha e^{i\vartheta^\alpha}=e^{it}-2\cos\gamma\,e^{i\epsilon}$, with $w\equiv\Delta c+is$ and $f\equiv1-(1+\Delta)c^2-i(1+\Delta)sc$. The key identity is

$$(1+\Delta)\Lambda^2\,Z=(1-\Delta)\,w\,f .$$

Since $(1-\Delta)/[(1+\Delta)\Lambda^2]>0$, this gives $\theta_{\rm M}=\operatorname{Arg}f+\operatorname{Arg}w$, which is Eq. (S3). Because $|w|=|f|=\Lambda$, it also gives $\lVert\mathbf M\rVert=(1-\Delta)/(1+\Delta)$.

In [7]:
Z = sum(cosT[a] + I*sinT[a] for a in 'ABC')
wf = D*c + I*s
ff = 1 - (1 + D)*c**2 - I*(1 + D)*s*c
check("(1+Delta) Lambda^2 Z = (1-Delta) w f", red((1 + D)*L**2*Z - (1 - D)*wf*ff))
check("|f|^2 = Lambda^2", red(ff*sp.conjugate(ff) - L**2))
check("|w|^2 = Lambda^2", red(wf*sp.conjugate(wf) - L**2))
check("|M|^2 = ((1-Delta)/(1+Delta))^2", red(Z*sp.conjugate(Z) - ((1 - D)/(1 + D))**2))

✓ (1+Delta) Lambda^2 Z = (1-Delta) w f
✓ |f|^2 = Lambda^2
✓ |w|^2 = Lambda^2
✓ |M|^2 = ((1-Delta)/(1+Delta))^2


**Three-to-one map.** Rewriting $f$ in exponential form,

$$f=\frac{1-\Delta}{2}-\frac{1+\Delta}{2}e^{2it}.$$

This is a circle of radius $(1+\Delta)/2$ centred at $(1-\Delta)/2$. The radius exceeds the distance to the origin, so $f$ encircles the origin twice per turn. Separately, $w$ encircles it once. Hence $\theta_{\rm M}$ winds three times while $\vartheta^{\rm A}$ winds once.

In [8]:
f_t = ff.subs({c: sp.cos(t), s: sp.sin(t)})
check("f = (1-Delta)/2 - (1+Delta)/2 e^{2it}",
      sp.expand_trig(sp.expand(sp.expand_complex(f_t - ((1 - D)/2 - (1 + D)/2*sp.exp(2*I*t))))))

tt = np.linspace(0, 2*np.pi, 20001)
for Dv in (0.1, 0.635, 0.95):
    fn = 1 - (1 + Dv)*np.cos(tt)**2 - 1j*(1 + Dv)*np.sin(tt)*np.cos(tt)
    wn = Dv*np.cos(tt) + 1j*np.sin(tt)
    wind = lambda v: (np.unwrap(np.angle(v))[-1] - np.unwrap(np.angle(v))[0])/(2*np.pi)
    print(f"Delta={Dv}: winding(f) = {wind(fn):.3f}, winding(w) = {wind(wn):.3f}, winding(theta_M) = {wind(fn*wn):.3f}")

✓ f = (1-Delta)/2 - (1+Delta)/2 e^{2it}
Delta=0.1: winding(f) = 2.000, winding(w) = 1.000, winding(theta_M) = 3.000
Delta=0.635: winding(f) = 2.000, winding(w) = 1.000, winding(theta_M) = 3.000
Delta=0.95: winding(f) = 2.000, winding(w) = 1.000, winding(theta_M) = 3.000


**Transverse-Y endpoint.** A short calculation gives $\operatorname{Re}(wf)=c\,[(1+\Delta)^2s^2-\Delta^2]$. Hence $M_\parallel=0$ exactly at $\sin\vartheta_0=\Delta/(1+\Delta)$, the endpoint quoted below Eq. (S1).

In [9]:
re_wf = sp.expand((wf*ff + sp.conjugate(wf*ff))/2)
check("Re(w f) = c[(1+Delta)^2 s^2 - Delta^2]", red(re_wf - c*((1 + D)**2*s**2 - D**2)))

✓ Re(w f) = c[(1+Delta)^2 s^2 - Delta^2]


### I.C The manifold rotation is not rigid, Eqs. (S4)–(S7)

**Jacobian.** For any complex $g$, $\frac{d}{dt}\operatorname{Arg}g=\operatorname{Im}(\dot g\bar g)/|g|^2$. Applied to the two factors of Eq. (S3):

In [10]:
def dArg(g):
    return red(((ddt(g)*sp.conjugate(g) - sp.conjugate(ddt(g))*g)/(2*I))/L**2)   # |g|^2 = Lambda^2 for g = f, w
dArg_f, dArg_w = dArg(ff), dArg(wf)
show("", sp.Symbol(r"\frac{d}{dt}\operatorname{Arg} f"), dArg_f)
show("", sp.Symbol(r"\frac{d}{dt}\operatorname{Arg} w"), dArg_w)
Jac = red(dArg_f + dArg_w)
check("J = 1 + 2 Delta / Lambda^2   (S4)", red(Jac - (1 + 2*D/L**2)))

Jac_t = 1 + 4*D/(1 + D**2 + (D**2 - 1)*sp.cos(2*t))
check("1 + 2Delta/Lambda^2 = 1 + 4Delta/(1+Delta^2+(Delta^2-1)cos 2t)",
      sp.simplify(sp.expand_trig((1 + 2*D/(sp.sin(t)**2 + D**2*sp.cos(t)**2)) - Jac_t)))
check("J = 3 at Delta = 1", Jac_t.subs(D, 1) - 3)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

✓ J = 1 + 2 Delta / Lambda^2   (S4)
✓ 1 + 2Delta/Lambda^2 = 1 + 4Delta/(1+Delta^2+(Delta^2-1)cos 2t)
✓ J = 3 at Delta = 1


**Sampling density, Eq. (S5).** We need $\int_0^{2\pi}\mathcal J\,dt$. By symmetry $\int_0^{2\pi}dt/\Lambda^2=4\int_0^{\pi/2}dt/(\sin^2t+\Delta^2\cos^2t)$, and the substitution $u=\tan t$ turns the right-hand side into $4\int_0^\infty du/(u^2+\Delta^2)$. Hence $\int_0^{2\pi}\mathcal J\,dt=6\pi$, so $P(\vartheta^{\rm A})=\mathcal J/6\pi$. The density $\mathcal J$ is largest where $\Lambda^2$ is smallest, i.e. $c^2=1$ ($\vartheta^{\rm A}=0,\pi$).

In [11]:
u = sp.Symbol('u', positive=True)
int_invL2 = 4*sp.integrate(1/(u**2 + D**2), (u, 0, sp.oo))
norm = 2*sp.pi + 2*D*int_invL2
show("(S5)", sp.Integral(sp.Symbol(r"\mathcal{J}"), (t, 0, 2*sp.pi)), sp.simplify(norm))

<IPython.core.display.Math object>

**Pullback metric, Eqs. (S6)–(S7).** Each angle derivative follows from $\dot\vartheta=\cos\vartheta\,\tfrac{d}{dt}\sin\vartheta-\sin\vartheta\,\tfrac{d}{dt}\cos\vartheta$. For $\epsilon$ and $\gamma$ the inputs are their $\cos$ and $\sin$ above.

In [12]:
eps_p = red(cos_e*ddt(sin_e) - sin_e*ddt(cos_e))
gam_p = red(-ddt(cos_g)/sin_g)
show("", sp.Symbol(r"\epsilon'"), eps_p)
show("", sp.Symbol(r"\gamma'"), gam_p)
thB_p = red(cosT['B']*ddt(sinT['B']) - sinT['B']*ddt(cosT['B']))
thC_p = red(cosT['C']*ddt(sinT['C']) - sinT['C']*ddt(cosT['C']))
check("dtheta^B/dt = eps' - gamma'", red(thB_p - (eps_p - gam_p)))
check("dtheta^C/dt = eps' + gamma'", red(thC_p - (eps_p + gam_p)))

g_metric = sp.Rational(1, 4)*(1 + thB_p**2 + thC_p**2)
S7 = sp.Rational(1, 4)*(1 + 2*D**2/L**4 + 2*D**2*(1 - D**2)**2*s**2*c**2/(L**4*G**2))
check("g = (1 + 2 eps'^2 + 2 gamma'^2)/4 = Eq. (S7)", red(g_metric - S7))
check("g -> 3/4 at Delta = 1", S7.subs({D: 1, L: 1, G: sp.sqrt(3)}) - sp.Rational(3, 4))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

✓ dtheta^B/dt = eps' - gamma'
✓ dtheta^C/dt = eps' + gamma'
✓ g = (1 + 2 eps'^2 + 2 gamma'^2)/4 = Eq. (S7)
✓ g -> 3/4 at Delta = 1


---
## II. Generalization of the spiral and its elastic signatures

### II.A Channels and their norms, Eqs. (S8)–(S10)

The channels are $\mathbf X^{(m)}=\frac1{\sqrt3}\sum_\alpha\omega^{m\alpha}\mathbf S_\alpha$ [Eq. (S9)]. Their Hermitian norms depend on the configuration only through the sum of the three pair products $d\equiv\sum_{\langle\alpha\beta\rangle}\mathbf S_\alpha\cdot\mathbf S_\beta$. Since $\lVert\mathbf M\rVert^2=3+2d$ is fixed on the manifold (Sec. I B), both norms are fixed too.

In [13]:
dAB, dBC, dCA = sp.symbols('d_AB d_BC d_CA', real=True)
gram = sp.Matrix([[1, dAB, dCA], [dAB, 1, dBC], [dCA, dBC, 1]])     # S_alpha . S_beta
def norm2(m):
    return sp.expand(sum(w**(m*a)*sp.conjugate(w)**(m*b)*gram[a, b] for a in range(3) for b in range(3))/3)
dsum = dAB + dBC + dCA
check("||X^(0)||^2 = (3 + 2 d)/3", norm2(0) - (3 + 2*dsum)/3)
check("||X^(+)||^2 = (3 - d)/3",   norm2(1) - (3 - dsum)/3)

d_manifold = (((1 - D)/(1 + D))**2 - 3)/2              # from |M|^2 = 3 + 2d
X0n2 = sp.factor((3 + 2*d_manifold)/3)
Xpn2 = sp.factor((3 - d_manifold)/3)
show("(S10)", sp.Symbol(r"\lVert\mathbf X^{(0)}\rVert^2"), X0n2)
show("(S10)", sp.Symbol(r"\lVert\mathbf X^{(+)}\rVert^2"), Xpn2)
check("||X^(+)||^2 = (2/3)(2+5D+2D^2)/(1+D)^2", Xpn2 - sp.Rational(2, 3)*(2 + 5*D + 2*D**2)/(1 + D)**2)
check("Parseval: ||X0||^2 + 2||X+||^2 = 3", X0n2 + 2*Xpn2 - 3)
vals = [float(Xpn2.subs(D, v)) for v in np.linspace(1e-6, 1, 2001)]
print(f"||X+||^2 over 0<Delta<=1: min {min(vals):.4f}, max {max(vals):.4f}, relative variation {(max(vals)-min(vals))/max(vals):.1%}")

✓ ||X^(0)||^2 = (3 + 2 d)/3
✓ ||X^(+)||^2 = (3 - d)/3


<IPython.core.display.Math object>

<IPython.core.display.Math object>

✓ ||X^(+)||^2 = (2/3)(2+5D+2D^2)/(1+D)^2
✓ Parseval: ||X0||^2 + 2||X+||^2 = 3
||X+||^2 over 0<Delta<=1: min 1.3333, max 1.5000, relative variation 11.1%


**Momentum of the channels, Eq. (S11).** With sublattice index $\alpha(\mathbf R)=(n_1+n_2)\bmod3$ and $\mathbf K=(1/3,1/3)$, we have $e^{2\pi i\mathbf K\cdot\mathbf R}=\omega^{\alpha(\mathbf R)}$. Inverting Eq. (S9) gives $\mathbf S_{\mathbf R}=\frac1{\sqrt3}\sum_m e^{-2\pi i m\mathbf K\cdot\mathbf R}\mathbf X^{(m)}$, so channel $m$ appears in $\mathbf M(\mathbf Q)$ only at $\mathbf Q\equiv-m\mathbf K$.

In [14]:
Lc = 6
R = [(n1, n2) for n1 in range(Lc) for n2 in range(Lc)]
Qs = {'Gamma': (0, 0), '+K': (1/3, 1/3), '-K': (-1/3, -1/3)}
om = np.exp(2j*np.pi/3)
for m in (1, 0, -1):
    amp = {k: abs(sum(om**(-m*((n1 + n2) % 3))*np.exp(-2j*np.pi*(Q[0]*n1 + Q[1]*n2)) for n1, n2 in R))
           for k, Q in Qs.items()}
    print(f"m={m:+d}: " + ", ".join(f"|sum| at {k} = {v:5.1f}" for k, v in amp.items()))

m=+1: |sum| at Gamma =   0.0, |sum| at +K =   0.0, |sum| at -K =  36.0
m=+0: |sum| at Gamma =  36.0, |sum| at +K =   0.0, |sum| at -K =   0.0
m=-1: |sum| at Gamma =   0.0, |sum| at +K =  36.0, |sum| at -K =   0.0


**Circular basis, Eq. (S12).** $X^{(m)}_{\pm}=(X^{(m)}_\parallel\pm iX^{(m)}_\perp)/\sqrt2=\frac1{\sqrt6}\sum_\alpha\omega^{m\alpha}e^{\pm i\vartheta^\alpha}$, and reality of the spins gives $X^{(-m)}_-=\overline{X^{(m)}_+}$.

In [15]:
thA, thB, thC = sp.symbols('vartheta_A vartheta_B vartheta_C', real=True)
ths = (thA, thB, thC)
def Xcirc(m, sgn):
    return sum(w**(m*a)*sp.exp(sgn*I*ths[a]) for a in range(3))/sp.sqrt(6)
def Xcart(m, comp):   # comp 0 = parallel (cos), 1 = perp (sin)
    f = sp.cos if comp == 0 else sp.sin
    return sum(w**(m*a)*f(ths[a]) for a in range(3))/sp.sqrt(3)
for m in (1, 0, -1):
    for sgn in (1, -1):
        check(f"X^({m})_{'+' if sgn > 0 else '-'} = (X_par {'+' if sgn > 0 else '-'} i X_perp)/sqrt2",
              sp.expand_complex(Xcirc(m, sgn) - (Xcart(m, 0) + sgn*I*Xcart(m, 1))/sp.sqrt(2)))
    check(f"X^({-m})_- = conj X^({m})_+", sp.expand_complex(Xcirc(-m, -1) - sp.conjugate(Xcirc(m, 1))))

✓ X^(1)_+ = (X_par + i X_perp)/sqrt2
✓ X^(1)_- = (X_par - i X_perp)/sqrt2
✓ X^(-1)_- = conj X^(1)_+
✓ X^(0)_+ = (X_par + i X_perp)/sqrt2
✓ X^(0)_- = (X_par - i X_perp)/sqrt2
✓ X^(0)_- = conj X^(0)_+
✓ X^(-1)_+ = (X_par + i X_perp)/sqrt2
✓ X^(-1)_- = (X_par - i X_perp)/sqrt2
✓ X^(1)_- = conj X^(-1)_+


### Series machinery

Perturbative results use a truncated power series in $\bar\Delta$ (class `Ser`). Each coefficient is a Laurent polynomial in $z=e^{it}$, and the basic operations work as follows:
- **Complex conjugation:** $i\to-i$, $z\to1/z$.
- **Derivative along the manifold:** $d/dt=iz\,d/dz$.
- **Average over one turn:** $\frac1{2\pi}\int_0^{2\pi}dt$ is the $z^0$ coefficient.
- **Powers and logarithms:** $(a_0+\delta)^p$ and $\log(a_0+\delta)$ are expanded in $\delta$. This requires the leading coefficient $a_0$ to be $t$-independent, which holds in every use below.

All series are kept through $\mathcal O(\bar\Delta^4)$. This one order beyond the tables is what Eq. (S34) needs.

In [16]:
x = sp.Symbol(r'\bar{\Delta}', positive=True)
z = sp.Symbol('z')
NMAX = 4

def zconj(e):
    return sp.expand(e.xreplace({I: -I}).subs(z, 1/z))

class Ser:
    # Truncated series sum_{k<=N} c_k(z) Deltabar^k.
    def __init__(self, coeffs, N=NMAX):
        self.N = N
        self.c = [sp.expand(coeffs[k]) if k < len(coeffs) else sp.Integer(0) for k in range(N + 1)]
    @staticmethod
    def of(expr, N=NMAX):
        e = sp.expand(expr)
        return Ser([e.coeff(x, k) for k in range(N + 1)], N)
    def _lift(self, o):
        return o if isinstance(o, Ser) else Ser([o], self.N)
    def __add__(self, o):
        o = self._lift(o); return Ser([a + b for a, b in zip(self.c, o.c)], self.N)
    __radd__ = __add__
    def __neg__(self): return Ser([-a for a in self.c], self.N)
    def __sub__(self, o): return self + (-self._lift(o))
    def __rsub__(self, o): return (-self) + o
    def __mul__(self, o):
        if not isinstance(o, Ser):
            return Ser([o*a for a in self.c], self.N)
        return Ser([sum(self.c[i]*o.c[k - i] for i in range(k + 1)) for k in range(self.N + 1)], self.N)
    __rmul__ = __mul__
    def conj(self): return Ser([zconj(a) for a in self.c], self.N)
    def ddt(self): return Ser([I*z*sp.diff(a, z) for a in self.c], self.N)
    def avg(self): return Ser([a.coeff(z, 0) if a.has(z) else a for a in self.c], self.N)
    def shift(self):           # divide by Deltabar (requires c_0 = 0)
        assert self.c[0] == 0; return Ser(self.c[1:], self.N - 1)
    def _split(self):
        a0 = self.c[0]; assert not a0.has(z), "leading coefficient must be t-independent"
        return a0, Ser([0] + self.c[1:], self.N)*(1/a0)
    def fpow(self, p):
        a0, d = self._split(); out, term = Ser([1], self.N), Ser([1], self.N)
        for k in range(1, self.N + 1):
            term = term*d; out = out + term*sp.binomial(p, k)
        return out*a0**p
    def flog(self):
        a0, d = self._split(); out, term = Ser([sp.log(a0)], self.N), Ser([1], self.N)
        for k in range(1, self.N + 1):
            term = term*d; out = out + term*(sp.Integer(-1)**(k + 1)/k)
        return out
    def im(self): return (self - self.conj())*(1/(2*I))
    def expr(self, upto=None):
        n = self.N if upto is None else upto
        return sum(self.c[k]*x**k for k in range(n + 1))

def to_trig(e):
    # Laurent polynomial in z = e^{it} -> trigonometric polynomial in t.
    coeffs = {}
    for term in sp.Add.make_args(sp.expand(e)):
        a, k = term.as_coeff_exponent(z)
        coeffs[int(k)] = coeffs.get(int(k), 0) + a
    out = coeffs.get(0, 0)
    for k in sorted({abs(k) for k in coeffs if k != 0}):
        ap, am = coeffs.get(k, 0), coeffs.get(-k, 0)
        out += sp.simplify(ap + am)*sp.cos(k*t) + sp.simplify(I*(ap - am))*sp.sin(k*t)
    return sp.expand(out)

def trig_series(S, upto=None):
    n = S.N if upto is None else upto
    return sum(to_trig(S.c[k])*x**k for k in range(n + 1))

The channels on the manifold, with $\Delta=1-\bar\Delta$, $c=(z+z^{-1})/2$, $s=(z-z^{-1})/2i$ and $e^{i\vartheta^{\rm B,C}}=-e^{i\epsilon}e^{\mp i\gamma}$. We work with $Y\equiv\sqrt6X$.

In [17]:
Dx = 1 - x
cz, sz = (z + 1/z)/2, (z - 1/z)/(2*I)
Lam2 = Ser.of(sz**2 + Dx**2*cz**2)
iLam = Lam2.fpow(-sp.Rational(1, 2))
cosg = Ser.of(Dx)*Ser([2, -1]).fpow(-1)*iLam                  # Delta/((1+Delta) Lambda)
sing = (1 - cosg*cosg).fpow(sp.Rational(1, 2))
e_eps = Ser.of(Dx*cz + I*sz)*iLam
eA = Ser.of(z)
eB = -e_eps*(cosg - I*sing)
eC = -e_eps*(cosg + I*sing)

def Yplus(m):                                # sqrt6 X^{(m)}_+
    return eA + eB*sp.expand(w**(m % 3)) + eC*sp.expand(w**((2*m) % 3))
Y0p, Y1p, Ym1p = Yplus(0), Yplus(1), Yplus(-1)
Y1m = Ym1p.conj()                            # sqrt6 X^{(+)}_- = conj(sqrt6 X^{(-)}_+)

for name, Y in (("sqrt6 X(+)_-", Y1m), ("sqrt6 X(+)_+", Y1p), ("sqrt6 X(0)_+", Y0p)):
    print(f"{name}:  O(Deltabar^0) = {Y.c[0]},   O(Deltabar^1) = {sp.factor(Y.c[1])}")

sqrt6 X(+)_-:  O(Deltabar^0) = 3/z,   O(Deltabar^1) = -(z - 1)*(z + 1)*(z**2 + 1)/(2*z**3)
sqrt6 X(+)_+:  O(Deltabar^0) = 0,   O(Deltabar^1) = 1/(2*z)
sqrt6 X(0)_+:  O(Deltabar^0) = 0,   O(Deltabar^1) = -z**3/2


$X^{(+)}_-=\tfrac{3}{\sqrt6}e^{-it}+\mathcal O(\bar\Delta)$ is the only component that survives at $\bar\Delta=0$: it is the dominant circular component.

**Chiral phase, Eq. (S13).** From $e^{-i\psi}=X^{(+)}_-/|X^{(+)}_-|$ and $\dot\psi=-\operatorname{Im}(\dot X^{(+)}_-\overline{X^{(+)}_-})/|X^{(+)}_-|^2$ we obtain two results:
- $\psi=\vartheta^{\rm A}+(\bar\Delta/3)\sin2\vartheta^{\rm A}+\mathcal O(\bar\Delta^2)$, quoted in Sec. II B;
- $|X^{(+)}_-|=\lVert\mathbf X^{(+)}\rVert+\mathcal O(\bar\Delta^2)$.

In [18]:
N2m = Y1m*Y1m.conj()                          # 6 |X(+)_-|^2
u_emipsi = Y1m*N2m.fpow(-sp.Rational(1, 2))   # e^{-i psi}
dpsi = (Y1m.ddt()*Y1m.conj()).im()*(-1)*N2m.fpow(-1)

dpsi_trig = trig_series(dpsi, 2)
show("", sp.Symbol(r"\dot\psi"), dpsi_trig + sp.O(x**3))
psi1 = sp.integrate(to_trig(dpsi.c[1]), t)
show("", sp.Symbol(r"\psi"), t + psi1*x + sp.O(x**2))

Xpn2_x = sp.series(Xpn2.subs(D, 1 - x), x, 0, 3).removeO()
check("|X(+)_-|^2 = ||X+||^2 + O(Deltabar^2)", sp.expand((N2m*(sp.Rational(1, 6))).expr(1) - Xpn2_x.coeff(x, 0) - Xpn2_x.coeff(x, 1)*x))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

✓ |X(+)_-|^2 = ||X+||^2 + O(Deltabar^2)


**Numerical check**: $\psi$ is monotonic and winds once per turn, for anisotropies across the full range.

In [19]:
def angles_num(tv, Dv):
    Lam = np.sqrt(np.sin(tv)**2 + Dv**2*np.cos(tv)**2)
    eps = np.unwrap(np.arctan2(np.sin(tv), Dv*np.cos(tv)))
    gam = np.arccos(Dv/((1 + Dv)*Lam))
    return np.array([tv, np.pi + eps - gam, np.pi + eps + gam])

def channels_num(tv, Dv):
    th = angles_num(tv, Dv)
    Xc = lambda m, sg: sum(om**(m*a)*np.exp(1j*sg*th[a]) for a in range(3))/np.sqrt(6)
    Xm = Xc(1, -1)
    psi = -np.unwrap(np.angle(Xm))
    return th, psi, {'X(+)_-': Xm, 'X(+)_+': Xc(1, 1), 'X(0)_+': Xc(0, 1)}

tv = np.linspace(0, 2*np.pi, 200001)
for Dv in (0.02, 0.3, 0.635, 0.95):
    th, psi, _ = channels_num(tv, Dv)
    dp = np.diff(psi)
    print(f"Delta={Dv}: min dpsi = {dp.min():.2e} (>0), psi(2pi)-psi(0) = {(psi[-1]-psi[0])/(2*np.pi):.6f} x 2pi")

Delta=0.02: min dpsi = 1.14e-05 (>0), psi(2pi)-psi(0) = 1.000000 x 2pi
Delta=0.3: min dpsi = 1.70e-05 (>0), psi(2pi)-psi(0) = 1.000000 x 2pi
Delta=0.635: min dpsi = 2.38e-05 (>0), psi(2pi)-psi(0) = 1.000000 x 2pi
Delta=0.95: min dpsi = 3.04e-05 (>0), psi(2pi)-psi(0) = 1.000000 x 2pi


### Spin-length constraint, Eqs. (S17)–(S18)

Insert the inverse of Eq. (S9), $\mathbf S_\alpha=\frac1{\sqrt3}\sum_m\omega^{-m\alpha}\mathbf X^{(m)}$, with $\mathbf X^{(-)}=\overline{\mathbf X^{(+)}}$ and a real $\mathbf X^{(0)}$. The squared length splits by powers of $\omega^{\alpha}$ as $|\mathbf S_\alpha|^2=C_0+\omega^{-\alpha}C_1+\omega^{\alpha}\bar C_1$. Requiring $|\mathbf S_\alpha|^2=1$ for all three $\alpha$ is therefore equivalent to $C_0=1$ and $C_1=0$, i.e. Eq. (S18).

In [20]:
a1, a2, b1, b2, g1, g2 = sp.symbols('a1 a2 b1 b2 g1 g2', real=True)
X0v = sp.Matrix([a1, a2])
Xpv = sp.Matrix([b1 + I*g1, b2 + I*g2])
Xmv = Xpv.conjugate()
dot = lambda u_, v_: sp.expand((u_.T*v_)[0])            # bilinear, no conjugation
Svec = [(X0v + sp.expand(w**(-a))*Xpv + sp.expand(w**a)*Xmv)/sp.sqrt(3) for a in range(3)]
len2 = [dot(v, v) for v in Svec]
C0 = sp.expand(sum(len2)/3)
C1 = sp.expand(sum(sp.expand(w**a)*len2[a] for a in range(3))/3)
check("C0 = (X0.X0 + 2 X+.X-)/3", C0 - (dot(X0v, X0v) + 2*dot(Xpv, Xmv))/3)
check("C1 = (2 X0.X+ + X-.X-)/3", C1 - (2*dot(X0v, Xpv) + dot(Xmv, Xmv))/3)
for a in range(3):
    check(f"|S_{a}|^2 = C0 + w^-a C1 + w^a conj(C1)",
          len2[a] - (C0 + sp.expand(w**(-a))*C1 + sp.expand(w**a)*sp.conjugate(C1)))

✓ C0 = (X0.X0 + 2 X+.X-)/3
✓ C1 = (2 X0.X+ + X-.X-)/3
✓ |S_0|^2 = C0 + w^-a C1 + w^a conj(C1)
✓ |S_1|^2 = C0 + w^-a C1 + w^a conj(C1)
✓ |S_2|^2 = C0 + w^-a C1 + w^a conj(C1)


On the manifold both conditions hold identically, and the cell below checks them order by order.

In the circular basis the bilinear product is $\mathbf a\cdot\mathbf b=a_+b_-+a_-b_+$, so with
$\mathbf X^{(-)}=\overline{\mathbf X^{(+)}}$ and a real $\mathbf X^{(0)}$,

$$\mathbf X^{(0)}\!\cdot\mathbf X^{(0)}=2|X^{(0)}_+|^2,\qquad
\mathbf X^{(+)}\!\cdot\mathbf X^{(-)}=|X^{(+)}_+|^2+|X^{(+)}_-|^2,$$

$$2\,\mathbf X^{(0)}\!\cdot\mathbf X^{(+)}+\mathbf X^{(-)}\!\cdot\mathbf X^{(-)}
=2\big(X^{(0)}_+X^{(+)}_-+\overline{X^{(0)}_+}X^{(+)}_+\big)+2\,\overline{X^{(+)}_-}\,\overline{X^{(+)}_+}.$$

These are the two expressions coded below. The factor $1/6$ undoes $Y\equiv\sqrt6\,X$, since every
term is quadratic in the channels.

**How to read the output.** Each quantity is a `Ser`, so it prints as a list whose entry $k$ is the
coefficient of $\bar\Delta^k$, and each entry is in general a Laurent polynomial in $z=e^{it}$. With
$S=1$ the expected answers are $[3,0,0,0,0]$ and $[0,0,0,0,0]$: a constant $3$ at order $\bar\Delta^0$
and nothing else, no $z$ anywhere. A violation would show up in one of two ways:

- a nonzero constant at order $\bar\Delta^k$ means the modulus budget is wrong at that order, as in the
  deliberately spoiled example below;
- a $z$-dependent entry means a violation that varies along the manifold, i.e. the spins do not have
  constant length.

The first condition is Parseval's identity: the weight is shared between the channels in the fixed
proportion of Eq. (S10), $\lVert\mathbf X^{(0)}\rVert^2+2\lVert\mathbf X^{(+)}\rVert^2=3S^2$, with no
$\bar\Delta$ corrections and no dependence on which member of the manifold the layer realizes. The
second is what later forces the $n=3$ harmonic of the uniform channel, Eq. (S26).

Both conditions are exact, not perturbative, so this is really a test of the series machinery — `Ser`,
the conjugation rule $z\to1/z$, and the channel construction — against an identity that must hold at
every order. The final lines confirm the same two conditions directly on the exact parametrization,
away from any expansion.

In [22]:
# the two combinations of Eq. (S18), built from the circular components (Y = sqrt6 X)
Parseval = (Y0p*Y0p.conj()*2 + (Y1p*Y1p.conj() + Y1m*Y1m.conj())*2)*(sp.Rational(1, 6))
C1ser    = ((Y0p*Y1m + Y0p.conj()*Y1p)*2 + Y1m.conj()*Y1p.conj()*2)*(sp.Rational(1, 6))

def report(name, S, expected):
    coeffs = [sp.simplify(cf) for cf in S.c]
    zdep = [k for k, cf in enumerate(coeffs) if cf.has(z)]
    print(f"{name}\n   coefficients of Deltabar^0..^{S.N}: {coeffs}"
          f"\n   residual t dependence at orders: {zdep if zdep else 'none'}   (expected value: {expected})")
    assert sp.simplify(S.expr() - expected) == 0

report("X0.X0 + 2 X+.X-   [real condition]   ", Parseval, 3)
report("2 X0.X+ + X-.X-   [complex condition]", C1ser, 0)

# what a violation looks like: rescale the uniform channel by 1 + Deltabar
Y0p_bad = Y0p*Ser([1, 1])
bad = (Y0p_bad*Y0p_bad.conj()*2 + (Y1p*Y1p.conj() + Y1m*Y1m.conj())*2)*(sp.Rational(1, 6))
print("\nwith X(0) rescaled by (1 + Deltabar):", [sp.simplify(cf) for cf in bad.c])

# the constraints are exact, not only order by order: spot check on the exact parametrization
for Dv in (0.9, 0.635, 0.2):
    tt = np.array([0.0, 0.7, 2.9, 5.1])
    thv = angles_num(tt, Dv)
    Xc = lambda m, sg: sum(om**(m*a)*np.exp(1j*sg*thv[a]) for a in range(3))/np.sqrt(6)
    X0p_, X1p_, X1m_ = Xc(0, 1), Xc(1, 1), Xc(1, -1)
    real_c = 2*abs(X0p_)**2 + 2*(abs(X1p_)**2 + abs(X1m_)**2)
    cplx_c = 2*(X0p_*X1m_ + np.conj(X0p_)*X1p_) + 2*np.conj(X1m_)*np.conj(X1p_)
    print(f"Delta={Dv}: max|real - 3| = {np.max(abs(real_c - 3)):.1e}, max|complex| = {np.max(abs(cplx_c)):.1e}")

X0.X0 + 2 X+.X-   [real condition]   
   coefficients of Deltabar^0..^4: [3, 0, 0, 0, 0]
   residual t dependence at orders: none   (expected value: 3)
2 X0.X+ + X-.X-   [complex condition]
   coefficients of Deltabar^0..^4: [0, 0, 0, 0, 0]
   residual t dependence at orders: none   (expected value: 0)

with X(0) rescaled by (1 + Deltabar): [3, 0, 0, 1/6, 1/4]
Delta=0.9: max|real - 3| = 1.3e-15, max|complex| = 6.5e-16
Delta=0.635: max|real - 3| = 1.8e-15, max|complex| = 5.7e-16
Delta=0.2: max|real - 3| = 1.3e-15, max|complex| = 8.7e-16


### Exact cyclic structure and selection rule, Eqs. (S19)–(S22)

Eq. (S19) follows from two symmetries that map the manifold onto itself:
- The relabeling $\mathbf S'_\alpha=\mathbf S_{\alpha+1}$ sends $X^{(m)}\to\omega^{-m}X^{(m)}$, hence $\psi\to\psi+2\pi/3$.
- The global reversal $\mathbf S\to-\mathbf S$ sends $\psi\to\psi+\pi$.

Because $\psi$ is a global coordinate on the manifold, the member at $\psi+2\pi/3$ is the relabeled one. Numerical check:

In [23]:
def wrapd(a): return np.max(np.abs((a + np.pi) % (2*np.pi) - np.pi))
Dv = 0.635
th, psi, _ = channels_num(tv, Dv)
err_perm = err_rev = 0
for i in rng.integers(0, len(tv) - 1, 20):
    for shift, kind in ((2*np.pi/3, 'perm'), (np.pi, 'rev')):
        target = (psi[i] + shift) % (2*np.pi)
        tp = np.interp(target, psi, tv)
        thp = angles_num(np.array([tp]), Dv)[:, 0]
        if kind == 'perm':   # S_alpha(psi + 2pi/3) = S_{alpha+1}(psi)
            err_perm = max(err_perm, wrapd(thp - th[[1, 2, 0], i]))
        else:                # S_alpha(psi + pi) = -S_alpha(psi)
            err_rev = max(err_rev, wrapd(thp - (th[:, i] + np.pi)))
print(f"max angle mismatch: permutation {err_perm:.1e}, reversal {err_rev:.1e}  (grid resolution ~{2*np.pi/len(tv):.0e})")

max angle mismatch: permutation 1.2e-10, reversal 2.7e-15  (grid resolution ~3e-05)


Expand $X^{(m)}=\sum_na^{(m)}_ne^{in\psi}$ [Eq. (S20)]. Combining Eq. (S21) with this expansion constrains each coefficient twice:
- the $2\pi/3$ shift requires $e^{2\pi in/3}=\omega^{-m}$, i.e. $n\equiv-m \pmod 3$;
- the $\pi$ shift requires $e^{i\pi n}=-1$, i.e. $n$ odd.

Together these give Eq. (S22) and the ladder of Eqs. (S23)–(S24).

In [24]:
for m in (1, 0, -1):
    allowed = [n for n in range(-11, 12)
               if sp.expand_complex(sp.exp(2*sp.pi*I*(n + m)/3)) == 1     # e^{2 pi i n/3} = omega^{-m}
               and sp.exp(I*sp.pi*n) == -1]                               # e^{i pi n} = -1
    print(f"m={m:+d}: allowed n = {allowed}   (n mod 6 = {sorted({n % 6 for n in allowed})})")

print("\nLadder Q_n = n (K, Q_z), in-plane part modulo reciprocal lattice:")
for n in (1, 3, 5, 7):
    h = sp.Rational(n, 3) % 1
    print(f"  n={n}: ({h}, {h}, {n} Q_z)")

m=+1: allowed n = [-7, -1, 5, 11]   (n mod 6 = [5])
m=+0: allowed n = [-9, -3, 3, 9]   (n mod 6 = [3])
m=-1: allowed n = [-11, -5, 1, 7]   (n mod 6 = [1])

Ladder Q_n = n (K, Q_z), in-plane part modulo reciprocal lattice:
  n=1: (1/3, 1/3, 1 Q_z)
  n=3: (0, 0, 3 Q_z)
  n=5: (2/3, 2/3, 5 Q_z)
  n=7: (1/3, 1/3, 7 Q_z)


### Locking of the uniform harmonic, Eqs. (S25)–(S26)

Keep only the fundamental $e^{-i\psi}$ in $X^{(+)}$ and a single harmonic $e^{\pm in\psi}$ in the real $X^{(0)}$. Then $\mathbf X^{(-)}\!\cdot\mathbf X^{(-)}\propto e^{2i\psi}$, while $\mathbf X^{(0)}\!\cdot\mathbf X^{(+)}$ carries $e^{i(n-1)\psi}$ and $e^{-i(n+1)\psi}$. Cancellation in Eq. (S18) requires one of these to be $e^{2i\psi}$, so $n=\pm3$, the same real harmonic pair.

In [25]:
psi_s = sp.Symbol('psi', real=True)
n_s = sp.Symbol('n', integer=True)
ap, am, beta = sp.symbols('alpha_+ alpha_- beta')
Xp_c = (ap*sp.exp(-I*psi_s), am*sp.exp(-I*psi_s))                                   # (X+_+, X+_-)
Xm_c = (sp.conjugate(am)*sp.exp(I*psi_s), sp.conjugate(ap)*sp.exp(I*psi_s))         # (X-_+, X-_-)
X0_c = (beta*sp.exp(I*n_s*psi_s), sp.conjugate(beta)*sp.exp(-I*n_s*psi_s))
bil = lambda A, B: A[0]*B[1] + A[1]*B[0]
def exponents(e):
    out = set()
    for term in sp.Add.make_args(sp.expand(e)):
        arg = sum(f.args[0] for f in sp.Mul.make_args(sp.powsimp(term)) if isinstance(f, sp.exp))
        out.add(sp.simplify(arg/(I*psi_s)))
    return sorted(out, key=str)
print("harmonics in X-.X- :", exponents(bil(Xm_c, Xm_c)))
print("harmonics in X0.X+ :", exponents(bil(X0_c, Xp_c)))
print("solutions for matching e^{2 i psi}:", sp.solve(sp.Eq(n_s - 1, 2), n_s), sp.solve(sp.Eq(-n_s - 1, 2), n_s))

harmonics in X-.X- : [2]
harmonics in X0.X+ : [-n - 1, n - 1]
solutions for matching e^{2 i psi}: [3] [-3]


**Table I** (elastic signatures) summarizes Eqs. (S14), (S22) and (S26):

| | rigid/screw stack | MI spiral |
|---|---|---|
| harmonics present | $n=0,\pm1$ only, if uniform | $n$ odd: $\pm1,\pm3,\pm5,\dots$ |
| unmodulated ($n=0$) weight | generically present | absent in every channel |
| longitudinal profile | common to all channels | locked to $m$ by Eq. (S22): $\Gamma$ at $3\times$ the $k_z$ of the $K$ channel |

### II.B Harmonic amplitudes, Eq. (S27) and Table II

Changing the integration variable from $\psi$ to $t$ in Eq. (S27) gives

$$a^{(m)}_{\varsigma,n}=\frac1{2\pi}\int_0^{2\pi}X^{(m)}_\varsigma(t)\,e^{-in\psi(t)}\,\dot\psi(t)\,dt=\big\langle X^{(m)}_\varsigma\,(e^{-i\psi})^n\,\dot\psi\big\rangle_t .$$

The average is the $z^0$ coefficient. All harmonics with $|n|\le9$ are computed through $\mathcal O(\bar\Delta^4)$.

In [26]:
powers = {0: Ser([1])}
u_epsi = u_emipsi.conj()
for n in range(1, 10):
    powers[n] = powers[n - 1]*u_emipsi          # e^{-i n psi}
    powers[-n] = powers[-(n - 1)]*u_epsi

comps = {'X(+)_-': Y1m, 'X(+)_+': Y1p, 'X(0)_+': Y0p}
acoef = {}
for name, Y in comps.items():
    YF = Y*dpsi
    for n in range(-9, 10):
        val = sp.factor(sp.expand((YF*powers[n]).avg().expr()/sp.sqrt(6)))
        if val != 0:
            acoef[(name, n)] = val
for (name, n), val in sorted(acoef.items(), key=lambda kv: (kv[0][0], kv[0][1])):
    chan, sg = name[1:4], name[-1]
    display(Math(rf"a^{{{chan}}}_{{{sg},{n:+d}}} = " + sp.latex(val) + r" + \mathcal{O}(\bar\Delta^5)"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Truncated to $\mathcal O(\bar\Delta^3)$, the entries reproduce Table II, and every other coefficient vanishes at that order. The expansion also gives the $\mathcal O(\bar\Delta^4)$ entries not listed in the table: $a^{(+)}_{+,-7}$, $a^{(0)}_{+,-3}$ and $a^{(0)}_{+,9}$.

In [27]:
def tr(e, k=3):
    e = sp.expand(e); return sp.expand(sum(e.coeff(x, j)*x**j for j in range(k + 1)))
r6 = sp.sqrt(6)
tableII = {('X(+)_-', -1): r6/72*(36 - x**2 - x**3),
           ('X(+)_+', -1): r6/216*x*(18 + 9*x + 5*x**2),
           ('X(0)_+',  3): -r6/48*x*(4 + 2*x + x**2),
           ('X(+)_-',  5): -r6/864*x**3,
           ('X(+)_+',  5): r6/72*x**2*(1 + x),
           ('X(+)_-', -7): -r6/864*x**3}
for key in sorted(set(acoef) | set(tableII), key=lambda k: (k[0], k[1])):
    check(f"a[{key[0]}, n={key[1]:+d}] matches Table II through O(Deltabar^3)",
          tr(acoef.get(key, 0)) - tr(tableII.get(key, 0)))
print("all coefficients real:", all(not v.has(I) for v in acoef.values()))

✓ a[X(+)_+, n=-7] matches Table II through O(Deltabar^3)
✓ a[X(+)_+, n=-1] matches Table II through O(Deltabar^3)
✓ a[X(+)_+, n=+5] matches Table II through O(Deltabar^3)
✓ a[X(+)_-, n=-7] matches Table II through O(Deltabar^3)
✓ a[X(+)_-, n=-1] matches Table II through O(Deltabar^3)
✓ a[X(+)_-, n=+5] matches Table II through O(Deltabar^3)
✓ a[X(0)_+, n=-3] matches Table II through O(Deltabar^3)
✓ a[X(0)_+, n=+3] matches Table II through O(Deltabar^3)
✓ a[X(0)_+, n=+9] matches Table II through O(Deltabar^3)
all coefficients real: True


**Numerical check** of the series at $\bar\Delta=0.05$ against quadrature of Eq. (S27) on the exact parametrization. The residual is the $\mathcal O(\bar\Delta^5)$ remainder, at most $\sim10^{-7}$.

In [30]:
trapz = getattr(np, "trapezoid", None) or np.trapz   # np.trapezoid is NumPy >= 2.0, np.trapz before
def harmonic_num(Dv, comp, n, M=400001):
    tv_ = np.linspace(0, 2*np.pi, M)
    _, psi_, X_ = channels_num(tv_, Dv)
    return trapz(X_[comp]*np.exp(-1j*n*psi_), psi_)/(2*np.pi)

xv = 0.05
worst = max(abs(harmonic_num(1 - xv, k[0], k[1]) - complex(v.subs(x, xv))) for k, v in acoef.items())
print(f"max |numeric - series| over all nonzero coefficients at Deltabar = {xv}: {worst:.1e}")

max |numeric - series| over all nonzero coefficients at Deltabar = 0.05: 7.0e-09


**Bragg weights, Eqs. (S28)–(S31).** $I_n=|a^{(m)}_{+,n}|^2+|a^{(m)}_{-,n}|^2$, with $a^{(m)}_{-,n}=a^{(m)}_{+,-n}$ for $m=0$. The fundamental uses the $m=+1$ channel.

In [31]:
A = lambda comp, n: acoef.get((comp, n), 0)
I_K  = tr(A('X(+)_-', -1)**2 + A('X(+)_+', -1)**2)
I_3Q = tr(A('X(0)_+', 3)**2 + A('X(0)_+', -3)**2)
show("(S28)", sp.Symbol(r"I_{\pm(\mathbf K,Q_z)}"), I_K + sp.O(x**4))
show("(S29)", sp.Symbol(r"I_{\pm(\boldsymbol\Gamma,3Q_z)}"), I_3Q + sp.O(x**4))
check("I_3Q = Deltabar^2 (4 + 2Deltabar + Deltabar^2)^2 / 384 through O(Deltabar^3)",
      I_3Q - tr(x**2*(4 + 2*x + x**2)**2/384))

# 5Q and 7Q from the Table II entries (the table's truncation order)
I_5Q = sp.expand(tableII[('X(+)_+', 5)]**2 + tableII[('X(+)_-', 5)]**2)
I_7Q = sp.expand(tableII[('X(+)_-', -7)]**2)
show("(S29)", sp.Symbol(r"I_{\pm5Q}"), sp.factor(I_5Q))
show("(S29)", sp.Symbol(r"I_{\pm7Q}"), I_7Q)
check("I_5Q = Deltabar^4 (1+Deltabar)^2/864 + Deltabar^6/124416", I_5Q - x**4*(1 + x)**2/864 - x**6/124416)

ratio_exact = (1 - D)**2/(4*(2 + 5*D + 2*D**2))
check("(S30): ratio = (1/2)||X0||^2/||X+||^2", ratio_exact - X0n2/(2*Xpn2))
show("(S30)", sp.Symbol(r"I_{3Q}/I_K"), sp.series(ratio_exact.subs(D, 1 - x), x, 0, 5))
show("(S31)", sp.Symbol(r"I_{5Q}/I_K"), sp.series(I_5Q/(sp.Rational(3, 2) - x**2/24 - x**3/24), x, 0, 6))
show("(S31)", sp.Symbol(r"I_{7Q}/I_K"), sp.series(I_7Q/(sp.Rational(3, 2) - x**2/24 - x**3/24), x, 0, 7))

Dv = 0.635; xv = 1 - Dv
IK_v = float(I_K.subs(x, xv))
print(f"\nDelta = {Dv}:  I_3Q/I_K (S30) = {float(ratio_exact.subs(D, Dv)):.2e},   I_5Q/I_K = {float(I_5Q.subs(x, xv))/IK_v:.1e}")

<IPython.core.display.Math object>

<IPython.core.display.Math object>

✓ I_3Q = Deltabar^2 (4 + 2Deltabar + Deltabar^2)^2 / 384 through O(Deltabar^3)


<IPython.core.display.Math object>

<IPython.core.display.Math object>

✓ I_5Q = Deltabar^4 (1+Deltabar)^2/864 + Deltabar^6/124416
✓ (S30): ratio = (1/2)||X0||^2/||X+||^2


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>


Delta = 0.635:  I_3Q/I_K (S30) = 5.57e-03,   I_5Q/I_K = 2.6e-05


### II.C Polarization of the harmonics, Eqs. (S32)–(S33)

Inverting Eq. (S12) at fixed $n$: $a_{\parallel,n}=(a_{+,n}+a_{-,n})/\sqrt2$, $a_{\perp,n}=(a_{+,n}-a_{-,n})/(i\sqrt2)$. For the fundamental both circular coefficients are real and positive, which gives the ellipse ratio of Eq. (S33). The exact value at $\Delta=0.635$ is computed by quadrature.

In [32]:
am1, ap1 = acoef[('X(+)_-', -1)], acoef[('X(+)_+', -1)]
ell = sp.series((am1 + ap1)/(am1 - ap1), x, 0, 3)
show("(S33)", sp.Symbol(r"|a^{(+)}_{\parallel,-1}|/|a^{(+)}_{\perp,-1}|"), ell)
check("ellipse ratio = 1 + Deltabar/3 + 2 Deltabar^2/9", ell.removeO() - (1 + x/3 + 2*x**2/9))

Dv = 0.635
am_n, ap_n = harmonic_num(Dv, 'X(+)_-', -1), harmonic_num(Dv, 'X(+)_+', -1)
rat = abs(am_n + ap_n)/abs(am_n - ap_n)
print(f"Delta = {Dv}: exact ratio = {rat:.3f}, intensity ratio easy-axis/transverse = {rat**2:.2f}")

<IPython.core.display.Math object>

✓ ellipse ratio = 1 + Deltabar/3 + 2 Deltabar^2/9
Delta = 0.635: exact ratio = 1.163, intensity ratio easy-axis/transverse = 1.35


**Circularity of the $3Q$ harmonic.** The two circular coefficients at $n=3$ are $a^{(0)}_{+,3}$ and $a^{(0)}_{-,3}=a^{(0)}_{+,-3}=\mathcal O(\bar\Delta^4)$. Their ratio is $-\bar\Delta^3/432$, so $|a^{(0)}_{\parallel,3}|/|a^{(0)}_{\perp,3}|=1-\bar\Delta^3/216+\dots$.

In [33]:
r3 = sp.series(acoef[('X(0)_+', -3)]/acoef[('X(0)_+', 3)], x, 0, 4)
show("", sp.Symbol(r"a^{(0)}_{-,3}/a^{(0)}_{+,3}"), r3)
circ = sp.series((1 + r3.removeO())/(1 - r3.removeO()), x, 0, 4)
show("", sp.Symbol(r"a^{(0)}_{\parallel,3}/a^{(0)}_{\perp,3}\ (\textrm{up to phase})"), circ)

for Dv in (0.98, 0.95, 0.9, 0.635):
    p3_, m3_ = harmonic_num(Dv, 'X(0)_+', 3), harmonic_num(Dv, 'X(0)_+', -3)
    dev_amp = abs(abs(p3_ + m3_)/abs(p3_ - m3_) - 1)
    print(f"Delta = {Dv}: exact |1 - |a_par|/|a_perp|| = {dev_amp:.2e},  Deltabar^3/216 = {(1-Dv)**3/216:.2e},  ratio = {dev_amp/((1-Dv)**3/216):.2f}")

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Delta = 0.98: exact |1 - |a_par|/|a_perp|| = 3.82e-08,  Deltabar^3/216 = 3.70e-08,  ratio = 1.03
Delta = 0.95: exact |1 - |a_par|/|a_perp|| = 6.25e-07,  Deltabar^3/216 = 5.79e-07,  ratio = 1.08
Delta = 0.9: exact |1 - |a_par|/|a_perp|| = 5.40e-06,  Deltabar^3/216 = 4.63e-06,  ratio = 1.17
Delta = 0.635: exact |1 - |a_par|/|a_perp|| = 4.19e-04,  Deltabar^3/216 = 2.25e-04,  ratio = 1.86


### II.D Layer-to-layer transport, Eqs. (S34)–(S37)

**Eq. (S34).** $X^{(0)}_+\propto e^{i\theta_{\rm M}}$ and $X^{(+)}_-\propto e^{-i\psi}$, so $\theta_{\rm M}-3\psi-\pi=\operatorname{Arg}\big(-X^{(0)}_+(X^{(+)}_-)^3\big)$. $X^{(0)}_+=\mathcal O(\bar\Delta)$, so the positive factor $\bar\Delta$ is divided out before taking the logarithm; this is why $X^{(0)}_+$ is needed through $\mathcal O(\bar\Delta^4)$.

The result is a series in $t$ whose first nonzero term is $\mathcal O(\bar\Delta^3)$. Replacing $t$ by $\psi=t+\mathcal O(\bar\Delta)$ therefore only affects $\mathcal O(\bar\Delta^4)$.

In [34]:
W = (-(Y0p*Y1m*Y1m*Y1m)).shift()
print("leading coefficient (must be t-independent and positive):", sp.nsimplify(W.c[0]))
gM = W.flog().im()
gM_trig = trig_series(gM, 3)
show("(S34)", sp.Symbol(r"\theta_{\rm M}-3\psi-\pi"), gM_trig + sp.O(x**4))
check("theta_M - 3 psi - pi = (Deltabar^3/216) sin 6t + O(Deltabar^4)", gM_trig - x**3*sp.sin(6*t)/216)

leading coefficient (must be t-independent and positive): 27/2


<IPython.core.display.Math object>

✓ theta_M - 3 psi - pi = (Deltabar^3/216) sin 6t + O(Deltabar^4)


**Eq. (S36).** $\phi_{l,+}-\phi_{l,-}=\operatorname{Arg}\big(X^{(+)}_+\overline{X^{(+)}_-}\big)$, with $\phi_{l,-}=-\psi_l$. Its first nonzero term is $\mathcal O(\bar\Delta)$, so only that order is unaffected by $t\to\psi$.

In [35]:
H = (Y1p*Y1m.conj()).shift()
print("leading coefficient:", sp.nsimplify(H.c[0]))
hphase = H.flog().im()
show("(S36)", sp.Symbol(r"\phi_{+}+\psi"), trig_series(hphase, 1) + sp.O(x**2))
check("phi_+ + psi = (Deltabar/6) sin 6t + O(Deltabar^2)", trig_series(hphase, 1) - x*sp.sin(6*t)/6)
check("ratio a(+)_{+,5}/a(+)_{+,-1} = Deltabar/6 + O(Deltabar^2)",
      tr(sp.series(acoef[('X(+)_+', 5)]/acoef[('X(+)_+', -1)], x, 0, 2).removeO(), 1) - x/6)

leading coefficient: 3/2


<IPython.core.display.Math object>

✓ phi_+ + psi = (Deltabar/6) sin 6t + O(Deltabar^2)
✓ ratio a(+)_{+,5}/a(+)_{+,-1} = Deltabar/6 + O(Deltabar^2)


**Eq. (S37).** The squared modulus of the subdominant component. Its $t$-dependence starts at $\mathcal O(\bar\Delta^3)$, so $t\to\psi$ is again harmless. The layer-independent $\bar\Delta^3/24$ inside the bracket also follows from Table II, $|a^{(+)}_{+,-1}|^2=\frac{\bar\Delta^2}{24}(1+\bar\Delta)+\dots$

In [36]:
Xpp2 = (Y1p*Y1p.conj())*(sp.Rational(1, 6))
Xpp2_trig = trig_series(Xpp2, 3)
show("(S37)", sp.Symbol(r"|X^{(+)}_{l,+}|^2"), Xpp2_trig + sp.O(x**4))
check("|X(+)_+|^2 = Deltabar^2/24 [1 + Deltabar + (Deltabar/3) cos 6t] + O(Deltabar^4)",
      Xpp2_trig - x**2/24*(1 + x + x*sp.cos(6*t)/3))

<IPython.core.display.Math object>

✓ |X(+)_+|^2 = Deltabar^2/24 [1 + Deltabar + (Deltabar/3) cos 6t] + O(Deltabar^4)


### II.E Energy minimization and determination of $Q_z$, Eqs. (S38)–(S48)

**Eqs. (S38)–(S39).** The interlayer matrix is circulant, so it is diagonalized by $v_m=(1,\omega^m,\omega^{2m})$.

In [37]:
J2, Ja, Jb = sp.symbols('J_2 J_a J_b', real=True)
Jmat = sp.Matrix([[J2, 3*Ja, 3*Jb], [3*Jb, J2, 3*Ja], [3*Ja, 3*Jb, J2]])
lam = {m: J2 + 3*sp.expand(w**(m % 3))*Ja + 3*sp.expand(w**((2*m) % 3))*Jb for m in (0, 1, -1)}
for m in (0, 1, -1):
    v = sp.Matrix([1, sp.expand(w**(m % 3)), sp.expand(w**((2*m) % 3))])
    assert sp.expand(Jmat*v - lam[m]*v) == sp.zeros(3, 1); print(f"\u2713 J v_{m} = lambda_{m} v_{m}")
show("(S39)", sp.Symbol(r"\lambda_0"), lam[0])
show("(S39)", sp.Symbol(r"\lambda_+"), lam[1])
check("lambda_- = conj(lambda_+)", sp.expand(lam[-1] - sp.conjugate(lam[1])))

✓ J v_0 = lambda_0 v_0
✓ J v_1 = lambda_1 v_1
✓ J v_-1 = lambda_-1 v_-1


<IPython.core.display.Math object>

<IPython.core.display.Math object>

✓ lambda_- = conj(lambda_+)


**Eq. (S40).** For arbitrary real in-plane spins in two adjacent layers, the bond energy is diagonal in the channel index.

In [38]:
Sl  = [sp.Matrix(sp.symbols(f'p{a}x p{a}y', real=True)) for a in range(3)]
Sl1 = [sp.Matrix(sp.symbols(f'q{a}x q{a}y', real=True)) for a in range(3)]
E_bond = sum(Jmat[a, b]*(Sl[a].T*Sl1[b])[0] for a in range(3) for b in range(3))
chan = lambda Sv, m: sum((sp.expand(w**((m*a) % 3))*Sv[a] for a in range(3)), sp.zeros(2, 1))/sp.sqrt(3)
X0l, X0l1 = chan(Sl, 0), chan(Sl1, 0)
Xpl, Xpl1 = chan(Sl, 1), chan(Sl1, 1)
hdot = (Xpl.T*Xpl1.conjugate())[0]
E_chan = lam[0]*(X0l.T*X0l1)[0] + lam[1]*hdot + sp.conjugate(lam[1]*hdot)
check("(S40): E_{l,l+1} = lambda_0 X0.X0' + 2 Re[lambda_+ X+.conj(X+')]", sp.expand(E_bond - E_chan))

✓ (S40): E_{l,l+1} = lambda_0 X0.X0' + 2 Re[lambda_+ X+.conj(X+')]


**Eqs. (S41)–(S43).** A real vector of fixed norm maps to the next layer by a rotation, and the Hermitian product separates on the circular basis.

In [39]:
n0, th1, th2 = sp.symbols('n_0 theta_1 theta_2', real=True)
v1 = n0*sp.Matrix([sp.cos(th1), sp.sin(th1)]); v2 = n0*sp.Matrix([sp.cos(th2), sp.sin(th2)])
check("(S41): X0.X0' = ||X0||^2 cos(delta theta_M)", sp.simplify((v1.T*v2)[0] - n0**2*sp.cos(th2 - th1)))

ua, ub, va, vb = sp.symbols('u_a u_b v_a v_b')     # generic complex in-plane components (par, perp)
aplus, aminus = (ua + I*ub)/sp.sqrt(2), (ua - I*ub)/sp.sqrt(2)
bplus, bminus = (va + I*vb)/sp.sqrt(2), (va - I*vb)/sp.sqrt(2)
check("(S42): a.conj(b) = a_+ conj(b_+) + a_- conj(b_-)",
      sp.expand(ua*sp.conjugate(va) + ub*sp.conjugate(vb) - aplus*sp.conjugate(bplus) - aminus*sp.conjugate(bminus)))

Rr, rho, m1, m2, dph = sp.symbols('R rho m_1 m_2 delta_phi', real=True)
term = Rr*sp.exp(I*rho)*m1*m2*sp.exp(-I*dph)
check("(S43): 2 Re[R e^{i rho} m m' e^{-i dphi}] = 2 R m m' cos(dphi - rho)",
      sp.simplify(sp.expand_complex(term + sp.conjugate(term)) - 2*Rr*m1*m2*sp.cos(dph - rho)))

✓ (S41): X0.X0' = ||X0||^2 cos(delta theta_M)
✓ (S42): a.conj(b) = a_+ conj(b_+) + a_- conj(b_-)
✓ (S43): 2 Re[R e^{i rho} m m' e^{-i dphi}] = 2 R m m' cos(dphi - rho)


**Bond energy, Eqs. (S44)–(S45), and the $\mathcal O(\bar\Delta^3)$ remainder of Eq. (S49).** Insert the transport relations into Eq. (S43):
- $\delta\theta_{{\rm M},l}=3\delta\psi_l+\frac{\bar\Delta^3}{216}(\sin6\psi_{l+1}-\sin6\psi_l)$ [Eq. S35], with $A=\lambda_0\lVert\mathbf X^{(0)}\rVert^2=\mathcal O(\bar\Delta^2)$;
- $\delta\phi_{l,-}=-\delta\psi_l$ exactly, and $\delta\phi_{l,+}=-\delta\psi_l+\frac{\bar\Delta}6(\sin6\psi_{l+1}-\sin6\psi_l)$ [Eq. S36];
- the moduli from Eq. (S37) as derived above, with $|X^{(+)}_{l,-}|^2=\lVert\mathbf X^{(+)}\rVert^2-|X^{(+)}_{l,+}|^2$.

The neglected $\mathcal O(\bar\Delta^2)$ term of Eq. (S36) multiplies $|X^{(+)}_+|^2=\mathcal O(\bar\Delta^2)$ and so enters only at $\mathcal O(\bar\Delta^4)$.

In [40]:
A2, Nn = sp.symbols('A_2 N', positive=True)        # A = A_2 Deltabar^2,  N = ||X+||^2 = O(1)
p1, p2, dq = sp.symbols('psi_l psi_l+1 delta_psi', real=True)
mp2 = lambda p: x**2/24*(1 + x) + x**3/72*sp.cos(6*p)
ds = sp.sin(6*p2) - sp.sin(6*p1)
bond = (A2*x**2*sp.cos(3*dq + x**3/216*ds)
        + 2*Rr*(sp.sqrt(mp2(p1))*sp.sqrt(mp2(p2))*sp.cos(dq + rho - x/6*ds)
                + sp.sqrt(Nn - mp2(p1))*sp.sqrt(Nn - mp2(p2))*sp.cos(dq + rho)))
bond3 = sp.expand(sp.series(bond, x, 0, 4).removeO())
B_ = 2*Rr*Nn
expected = A2*x**2*sp.cos(3*dq) + B_*sp.cos(dq + rho) + Rr*x**3/72*ds*sp.sin(dq + rho)
show("one bond [Eqs. (S44), (S45), (S49)]", sp.Symbol(r"E_{l,l+1}"), sp.collect(bond3, x))
check("E_{l,l+1} = A cos 3dpsi + B cos(dpsi+rho) + (R Db^3/72)(sin6psi_{l+1}-sin6psi_l) sin(rho+dpsi) + O(Db^4)",
      sp.simplify(bond3 - expected))

<IPython.core.display.Math object>

✓ E_{l,l+1} = A cos 3dpsi + B cos(dpsi+rho) + (R Db^3/72)(sin6psi_{l+1}-sin6psi_l) sin(rho+dpsi) + O(Db^4)


The first two terms are $E(\delta\psi_l)$ of Eq. (S45), so each bond depends only on its own increment, Eq. (S44). Under the uniform-advance ansatz $\delta\psi_l=q$ every bond has energy $E(q)$, and the $\mathcal O(\bar\Delta^3)$ remainder telescopes over any periodic stack, because $\psi_N=\psi_0+2\pi p$ implies $\sin6\psi_N=\sin6\psi_0$.

**Eqs. (S46)–(S48).** At $\Delta=1$, $A=0$ and the minimum of $B\cos(q+\rho)$ is at $q_0=\pi-\rho$. For $\bar\Delta>0$, linearize the stationarity condition around $q_0$.

In [41]:
lamp = lam[1]
Re_l, Im_l = sp.simplify((lamp + sp.conjugate(lamp))/2), sp.simplify((lamp - sp.conjugate(lamp))/(2*I))
tan_q0 = sp.simplify(-Im_l/Re_l)                               # tan(pi - rho) = -tan(rho)
show("(S46)", sp.Symbol(r"\tan 2\pi Q_z^{(0)}"), tan_q0)
check("(S46) = 3 sqrt3 (Jb - Ja)/(2J2 - 3(Ja + Jb))", sp.simplify(tan_q0 - 3*sp.sqrt(3)*(Jb - Ja)/(2*J2 - 3*(Ja + Jb))))

q, dqq, A_, B = sp.symbols('q delta_q A B', real=True)
q0 = sp.Symbol('q_0', real=True)
E_q = A_*sp.cos(3*q) + B*sp.cos(q + rho)
stat = sp.diff(E_q, q).subs(q, q0 + dqq).subs(rho, sp.pi - q0)
stat_lin = sp.series(stat, dqq, 0, 2).removeO()
stat_lin = sp.expand(stat_lin)
dq_sol = sp.solve(stat_lin.coeff(dqq, 0) + stat_lin.coeff(dqq, 1).subs(A_, 0)*dqq, dqq)[0]   # A*delta_q is O(Deltabar^4)
check("delta q = 3 A sin(3 q0)/B", dq_sol - 3*A_*sp.sin(3*q0)/B)
show("", sp.Symbol(r"\delta q"), dq_sol)

R_expr = sp.sqrt(sp.expand(Re_l**2 + Im_l**2))
check("R = |lambda_+| = (1/2) sqrt((2J2 - 3(Ja+Jb))^2 + 27(Jb-Ja)^2)",
      sp.expand(R_expr**2 - ((2*J2 - 3*(Ja + Jb))**2 + 27*(Jb - Ja)**2)/4))
A_lead = sp.series(lam[0]*X0n2.subs(D, 1 - x), x, 0, 3).removeO()          # lambda_0 Db^2/12
B_lead = sp.series(2*Xpn2.subs(D, 1 - x), x, 0, 1).removeO()                # 3 (times R)
pref = sp.simplify((3*A_lead/(B_lead*sp.Symbol('R'))).subs(sp.Symbol('R'), sp.sqrt((2*J2 - 3*(Ja + Jb))**2 + 27*(Jb - Ja)**2)/2)/x**2)
show("(S47)", sp.Symbol(r"(2\pi Q_z - 2\pi Q_z^{(0)})/[\bar\Delta^2 \sin 6\pi Q_z^{(0)}]"), pref)
check("(S47) prefactor", sp.simplify(pref - (J2 + 3*Ja + 3*Jb)/(6*sp.sqrt((2*J2 - 3*(Ja + Jb))**2 + 27*(Jb - Ja)**2))))

bound = sp.nsimplify(pref.subs({Jb: 0, Ja: 1, J2: 1}))
show("(S48)", sp.Symbol(r"\max_{J_b=0,\,J_a=J_2}|2\pi Q_z-2\pi Q_z^{(0)}|/\bar\Delta^2"), bound)
print(f"= {float(bound):.3f};   |delta Q_z| at Delta = 0.635: {float(bound)*0.365**2/(2*np.pi):.1e}")

<IPython.core.display.Math object>

✓ (S46) = 3 sqrt3 (Jb - Ja)/(2J2 - 3(Ja + Jb))
✓ delta q = 3 A sin(3 q0)/B


<IPython.core.display.Math object>

✓ R = |lambda_+| = (1/2) sqrt((2J2 - 3(Ja+Jb))^2 + 27(Jb-Ja)^2)


<IPython.core.display.Math object>

✓ (S47) prefactor


<IPython.core.display.Math object>

= 0.126;   |delta Q_z| at Delta = 0.635: 2.7e-03


### II.F Range of validity of the uniform-advance ansatz, Eqs. (S49)–(S54)

**Stationarity of a uniform stack.** Through $\mathcal O(\bar\Delta^2)$ the total energy is $\sum_lE(\delta\psi_l)$. The coordinate $\psi_m$ enters only $\delta\psi_{m-1}$ and $\delta\psi_m$, so

$$\partial_{\psi_m}\sum_lE(\delta\psi_l)=E'(\delta\psi_{m-1})-E'(\delta\psi_m).$$

This vanishes for equal increments, for any function $E$. At $\mathcal O(\bar\Delta^3)$ the cancellation fails, through the three contributions listed above Eq. (S44).

In [42]:
Ef = sp.Function('E')
psis = sp.symbols('psi_{m-2} psi_{m-1} psi_m psi_{m+1} psi_{m+2}', real=True)
tot = sum(Ef(psis[k + 1] - psis[k]) for k in range(4))
grad_m = sp.diff(tot, psis[2]).doit()
psi0 = sp.Symbol('psi_0', real=True)
uni = {psis[k]: psi0 + (k - 2)*q for k in range(5)}             # psi_m = psi_0
check("vanishes on a uniform stack for arbitrary E", sp.simplify(grad_m.subs(uni).doit()))

✓ vanishes on a uniform stack for arbitrary E


**Elastic energy and source term, Eqs. (S50)–(S52).** Write $\psi_l=\bar\psi_l+u_l$ with $\bar\psi_l=\psi_0+ql$. Then:
- **Elastic energy.** It is quadratic in $u$, $\tfrac12E''(q)\sum_l(u_{l+1}-u_l)^2$; its derivative with respect to $\psi_m$ is linear in $u$.
- **Source term.** The derivative of the $\mathcal O(\bar\Delta^3)$ sum of Eq. (S49), evaluated on the uniform stack, has the same discrete-Laplacian shape.

In [43]:
us = sp.symbols('u_{m-1} u_m u_{m+1}', real=True)
Ep = sp.symbols("E_0 E_1 E_2", real=True)                 # E(q), E'(q), E''(q)
E_taylor = lambda d: Ep[0] + Ep[1]*d + Ep[2]*d**2/2
elastic = E_taylor(us[1] - us[0]) + E_taylor(us[2] - us[1])   # the two bonds containing u_m
check("elastic derivative = E''(q) (2u_m - u_{m+1} - u_{m-1}) + [telescoping E'(q) terms]",
      sp.expand(sp.diff(elastic, us[1]) - Ep[2]*(2*us[1] - us[2] - us[0])))

cc = sp.Symbol('c_3', positive=True)                         # c_3 = R Deltabar^3/72
src = sum(cc*(sp.sin(6*psis[k + 1]) - sp.sin(6*psis[k]))*sp.sin(rho + psis[k + 1] - psis[k]) for k in range(4))
dsrc = sp.simplify(sp.diff(src, psis[2]).subs(uni))
target = cc*sp.cos(rho + q)*(2*sp.sin(6*uni[psis[2]]) - sp.sin(6*uni[psis[3]]) - sp.sin(6*uni[psis[1]]))
check("source derivative = c_3 cos(rho+q) (2 sin6psi_m - sin6psi_{m+1} - sin6psi_{m-1})",
      trig_zero(dsrc - target))

pb = sp.Symbol(r'\bar\psi_m', real=True)
lap = 2*sp.sin(6*pb) - sp.sin(6*pb + 6*q) - sp.sin(6*pb - 6*q)
check("(S50): 2 sin6psi_m - sin6psi_{m+1} - sin6psi_{m-1} = 2(1 - cos6q) sin6psi_m",
      trig_zero(lap - 2*(1 - sp.cos(6*q))*sp.sin(6*pb)))

✓ elastic derivative = E''(q) (2u_m - u_{m+1} - u_{m-1}) + [telescoping E'(q) terms]
✓ source derivative = c_3 cos(rho+q) (2 sin6psi_m - sin6psi_{m+1} - sin6psi_{m-1})
✓ (S50): 2 sin6psi_m - sin6psi_{m+1} - sin6psi_{m-1} = 2(1 - cos6q) sin6psi_m


With $u_l=u\sin6\bar\psi_l$, both contributions carry $2(1-\cos6q)\sin6\bar\psi_m$, and stationarity reduces to Eq. (S51): $uE''(q)+c_3\cos(\rho+q)=0$. Here $E''(q)=-9A\cos3q-B\cos(q+\rho)$, with $A=\mathcal O(\bar\Delta^2)$ and $B=2R\lVert\mathbf X^{(+)}\rVert^2=3R+\mathcal O(\bar\Delta^2)$.

In [44]:
u_amp = sp.Symbol('u')
Epp = -9*A_lead*sp.cos(3*q) - 2*Rr*sp.series(Xpn2.subs(D, 1 - x), x, 0, 3).removeO()*sp.cos(q + rho)
u_sol = sp.solve(sp.Eq(u_amp*Epp + Rr*x**3/72*sp.cos(rho + q), 0), u_amp)[0]
show("(S52)", sp.Symbol("u"), sp.series(u_sol, x, 0, 4))
check("u = Deltabar^3/216 + O(Deltabar^4)", sp.series(u_sol, x, 0, 4).removeO() - x**3/216)

<IPython.core.display.Math object>

✓ u = Deltabar^3/216 + O(Deltabar^4)


**Eqs. (S53)–(S54).** Insert $\psi_l=\bar\psi_l+\frac{\bar\Delta^3}{216}\sin6\bar\psi_l$ into Eq. (S34). This gives $\theta_{{\rm M},l}$, and the layer step follows from sum-to-product.

In [45]:
pbl = sp.Symbol(r'\bar\psi_l', real=True)
thetaM = lambda psi_: 3*psi_ + sp.pi + x**3/216*sp.sin(6*psi_)
psi_mod = lambda pb_: pb_ + x**3/216*sp.sin(6*pb_)
thM_l = sp.series(thetaM(psi_mod(pbl)), x, 0, 4).removeO()
check("(S53): theta_M,l = 3 psibar_l + pi + (Deltabar^3/54) sin 6 psibar_l", sp.expand(thM_l - (3*pbl + sp.pi + x**3/54*sp.sin(6*pbl))))
dthM = sp.series(thetaM(psi_mod(pbl + q)) - thetaM(psi_mod(pbl)), x, 0, 4).removeO()
check("(S53): delta theta_M,l = 3q + (Deltabar^3/27) sin(3q) cos(6 psibar_l + 3q)",
      trig_zero(dthM - (3*q + x**3/27*sp.sin(3*q)*sp.cos(6*pbl + 3*q))))

Dv, Qz = 0.635, 20/109
b54 = (1 - Dv)**3/27*abs(np.sin(6*np.pi*Qz))
print(f"(S54) bound at Delta = {Dv}, Q_z = 20/109: {b54:.2e} rad = {np.degrees(b54):.3f} deg per layer")

✓ (S53): theta_M,l = 3 psibar_l + pi + (Deltabar^3/54) sin 6 psibar_l
✓ (S53): delta theta_M,l = 3q + (Deltabar^3/27) sin(3q) cos(6 psibar_l + 3q)
(S54) bound at Delta = 0.635, Q_z = 20/109: 5.61e-04 rad = 0.032 deg per layer


### Numerical check of Eqs. (S52)–(S54)

Minimize the interlayer energy, Eq. (S38), over one manifold coordinate per layer. The setup is a periodic stack of $N=109$ layers with winding $20$, $J_b=0$, and $J_2/J_a$ set by Eq. (S46) to give $Q_z^{(0)}=20/109$. Restricting to the manifold makes the result independent of the overall scale of the interlayer couplings. Newton iterations use the analytic gradient.

In [46]:
Nl, pw = 109, 20
q0v = 2*np.pi*pw/Nl
J2v = 1.5 + 1.5*np.sqrt(3)/np.tan(np.pi - q0v)          # Ja = 1, Jb = 0
Jv = np.array([[J2v, 3, 0], [0, J2v, 3], [3, 0, J2v]])

def lay(tl, Dv):
    Lam = np.sqrt(np.sin(tl)**2 + Dv**2*np.cos(tl)**2)
    eps = tl + np.arctan2((1 - Dv)*np.sin(tl)*np.cos(tl), Dv*np.cos(tl)**2 + np.sin(tl)**2)
    cg = Dv/(1 + Dv)
    gam = np.arccos(cg/Lam)
    dLam = np.sin(tl)*np.cos(tl)*(1 - Dv**2)/Lam
    deps, dgam = Dv/Lam**2, cg*dLam/(Lam*np.sqrt(Lam**2 - cg**2))
    return (np.array([tl, np.pi + eps - gam, np.pi + eps + gam]),
            np.array([np.ones_like(tl), deps - dgam, deps + dgam]))

def grad_E(tl, Dv):
    th, dth = lay(np.append(tl, tl[0] + 2*np.pi*pw), Dv)
    sn = Jv[:, :, None]*np.sin(th[:, None, :-1] - th[None, :, 1:])
    g = np.zeros(Nl + 1)
    g[:-1] += np.sum(-sn.sum(1)*dth[:, :-1], 0)
    g[1:] += np.sum(sn.sum(0)*dth[:, 1:], 0)
    g[0] += g[-1]
    return g[:-1]

def minimize_stack(Dv, iters=10):
    tg = np.linspace(-0.5, 2*np.pi*pw + 0.5, 1000001)
    _, psig, _ = channels_num(tg, Dv)
    psig -= psig[np.argmin(abs(tg))]
    tl = np.interp(0.1 + q0v*np.arange(Nl), psig, tg)          # uniform-psi start
    for _ in range(iters):
        g = grad_E(tl, Dv)
        H = np.array([(grad_E(tl + 1e-5*np.eye(Nl)[k], Dv) - grad_E(tl - 1e-5*np.eye(Nl)[k], Dv))/2e-5 for k in range(Nl)]).T
        tl = tl - np.linalg.pinv(0.5*(H + H.T), rcond=1e-10) @ g
    return tl, np.max(abs(grad_E(tl, Dv)))

wrap = lambda a: (a + np.pi) % (2*np.pi) - np.pi
ll = np.arange(Nl)
for Dv in (0.99, 0.98, 0.96, 0.635):
    tl, gmax = minimize_stack(Dv)
    th, _ = lay(np.append(tl, tl[0] + 2*np.pi*pw), Dv)
    psi_l = -np.unwrap(np.angle(sum(om**a*np.exp(-1j*th[a]) for a in range(3))))[:-1]
    a0 = np.mean(psi_l - q0v*ll); sb = np.sin(6*(a0 + q0v*ll))
    u_fit = np.sum((psi_l - a0 - q0v*ll)*sb)/np.sum(sb**2)
    thM = np.arctan2(np.sin(th).sum(0), np.cos(th).sum(0))
    dev = np.max(abs(wrap(np.diff(thM) - 3*q0v)))
    x_ = 1 - Dv
    b54v = x_**3/27*abs(np.sin(3*q0v))
    print(f"Delta={Dv:5.3f} |grad|={gmax:.0e}:  u/Db^3 = {u_fit/x_**3:.5f} (1/216 = {1/216:.5f});  "
          f"max|dtheta_M - 6 pi Qz| = {np.degrees(dev):.2e} deg, (S54) = {np.degrees(b54v):.2e} deg, ratio {dev/b54v:.2f}")

Delta=0.990 |grad|=2e-13:  u/Db^3 = 0.00470 (1/216 = 0.00463);  max|dtheta_M - 6 pi Qz| = 6.71e-07 deg, (S54) = 6.62e-07 deg, ratio 1.01
Delta=0.980 |grad|=2e-13:  u/Db^3 = 0.00477 (1/216 = 0.00463);  max|dtheta_M - 6 pi Qz| = 5.45e-06 deg, (S54) = 5.29e-06 deg, ratio 1.03
Delta=0.960 |grad|=2e-13:  u/Db^3 = 0.00492 (1/216 = 0.00463);  max|dtheta_M - 6 pi Qz| = 4.50e-05 deg, (S54) = 4.23e-05 deg, ratio 1.06
Delta=0.635 |grad|=3e-13:  u/Db^3 = 0.00800 (1/216 = 0.00463);  max|dtheta_M - 6 pi Qz| = 5.70e-02 deg, (S54) = 3.22e-02 deg, ratio 1.77


$u/\bar\Delta^3\to1/216$ as $\bar\Delta\to0$, with a relative correction $\simeq1.6\bar\Delta$, as expected for an $\mathcal O(\bar\Delta^4)$ remainder. The last column quantifies the closing statement of Sec. II F: at $\Delta=0.635$ the bound of Eq. (S54) gives $0.03^\circ$ per layer while the actual step deviation is $0.06^\circ$, so the perturbative estimate is low by a factor $\approx1.8$ there and the residual layer dependence is indeed small.